# Part 1: 실험 준비 — 데이터 & 간단 평가셋 만들기

이 섹션에서는 다음을 다룹니다:
1. **데이터 소개**: IT_IBKS20260130IBK투자증권 PDF 구조
2. **질문 리스트 설계**: 핵심 요약/수치 질의/표 기반 질의 등 카테고리
3. **정답셋 + 평가 함수**: 기존 질문·정답 사용, 정확도·근거성·속도 평가

## 1-1. 데이터 소개: IT_IBKS20260130IBK투자증권 PDF 구조

이 PDF는 IBK투자증권이 작성한 삼성전자·SK하이닉스 리포트입니다. RAG 실험에 적합한 이유:
- **목차**: 삼성전자/SK하이닉스 실적, Check Point, 투자의견 등 구조화된 구성
- **표**: 분기별 실적 추이, 사업부별 매출/영업이익 등 표가 많음 (행·열 구조 중요)
- **주석·각주**: 단위(십억원, 조원), 출처 등
- **페이지 구성**: 15~26페이지 수준, 표가 여러 페이지에 걸쳐 있음

In [1]:
# PDF 기본 정보 확인
import fitz
pdf_path = "IT_IBKS20260130IBK투자증권.pdf"

doc = fitz.open(pdf_path)
print(f"페이지 수: {len(doc)}")
print(f"메타데이터: {doc.metadata}")
doc.close()

페이지 수: 26
메타데이터: {'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Microsoft® Word 2016', 'producer': 'Microsoft® Word 2016', 'creationDate': "D:20260130070152+09'00'", 'modDate': "D:20260130070152+09'00'", 'trapped': '', 'encryption': None}


In [2]:
# 페이지별 텍스트 샘플 (1페이지)
import fitz
doc = fitz.open(pdf_path)
page0 = doc[0]
text = page0.get_text()[:500]
print(text)
doc.close()

## 1-2. 기본 질문 리스트 설계

RAG 평가를 위한 질문은 다음 유형으로 설계할 수 있습니다:
- **핵심 요약**: "삼성전자 2025년 4분기 실적 요약은?"
- **수치 질의**: "매출액", "영업이익", "목표주가" 등 구체적 수치
- **표 기반 질의**: 표에서 특정 행·열 교차점 값을 묻는 질문
- **리스크 질의**: 투자 리스크, 전망 등

본 실험에서는 **재무(수치)**, **투자의견**, **비교** 카테고리의 12개 질문을 사용합니다.

In [3]:
# 기존 12개 질문의 카테고리 분포
from collections import Counter

test_questions = [
    {"question": "삼성전자 2025년 4분기 매출액은?", "expected_answer": "93.8조원", "category": "재무"},
    {"question": "삼성전자 2025년 4분기 영업이익은?", "expected_answer": "20.1조원", "category": "재무"},
    {"question": "삼성전자 DS 부문 2025년 4분기 영업이익은?", "expected_answer": "16.4조원", "category": "재무"},
    {"question": "삼성전자 메모리 사업부 2025년 4분기 영업이익은?", "expected_answer": "17.2조원", "category": "재무"},
    {"question": "SK하이닉스 2025년 4분기 영업이익은?", "expected_answer": "19.2조원", "category": "재무"},
    {"question": "2026년 1분기 삼성전자 영업이익은?", "expected_answer": "35.3조원", "category": "재무"},
    {"question": "2026년 1분기 SK하이닉스 영업이익은?", "expected_answer": "31조원", "category": "재무"},
    {"question": "삼성전자 목표주가는?", "expected_answer": "240,000원", "category": "투자의견"},
    {"question": "SK하이닉스 목표주가는?", "expected_answer": "1,100,000원", "category": "투자의견"},
    {"question": "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?", "expected_answer": "SK하이닉스", "category": "비교"},
    {"question": "2026년 1분기 삼성전자 메모리 부문 영업이익은?", "expected_answer": "33.1조원", "category": "재무"},
    {"question": "2025년 4분기 DRAM ASP 상승률은 삼성과 SK 중 어느 쪽이 더 높은가?", "expected_answer": "삼성전자", "category": "비교"},
]

cats = Counter(t["category"] for t in test_questions)
for c, n in cats.items():
    print(f"{c}: {n}개")

재무: 8개
투자의견: 2개
비교: 2개


## 1-3. 정답셋 만들기 (기존 질문/정답 사용)

평가 항목:
- **정확도**: expected_answer가 LLM 답변에 포함되는지 (전체 일치 또는 부분 일치)
- **근거성**: 인용된(retrieved) 문서에 expected_answer 또는 관련 수치가 포함되는지 (Yes/No)
- **속도**: 응답 시간(초)

In [4]:
# IBK투자증권 리포트 기반 정답셋 (12개) — keywords 포함
test_questions = [
    {"question": "삼성전자 2025년 4분기 매출액은?", "keywords": ["삼성전자", "2025", "4분기", "매출액"], "expected_answer": "93.8조원", "category": "재무"},
    {"question": "삼성전자 2025년 4분기 영업이익은?", "keywords": ["삼성전자", "2025", "4분기", "영업이익"], "expected_answer": "20.1조원", "category": "재무"},
    {"question": "삼성전자 DS 부문 2025년 4분기 영업이익은?", "keywords": ["DS", "영업이익", "2025"], "expected_answer": "16.4조원", "category": "재무"},
    {"question": "삼성전자 메모리 사업부 2025년 4분기 영업이익은?", "keywords": ["메모리", "영업이익", "2025"], "expected_answer": "17.2조원", "category": "재무"},
    {"question": "SK하이닉스 2025년 4분기 영업이익은?", "keywords": ["SK하이닉스", "영업이익", "2025"], "expected_answer": "19.2조원", "category": "재무"},
    {"question": "2026년 1분기 삼성전자 영업이익은?", "keywords": ["2026", "삼성전자", "영업이익"], "expected_answer": "35.3조원", "category": "재무"},
    {"question": "2026년 1분기 SK하이닉스 영업이익은?", "keywords": ["2026", "SK하이닉스", "영업이익"], "expected_answer": "31조원", "category": "재무"},
    {"question": "삼성전자 목표주가는?", "keywords": ["삼성전자", "목표주가"], "expected_answer": "240,000원", "category": "투자의견"},
    {"question": "SK하이닉스 목표주가는?", "keywords": ["SK하이닉스", "목표주가"], "expected_answer": "1,100,000원", "category": "투자의견"},
    {"question": "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?", "keywords": ["25년", "4분기", "영업이익"], "expected_answer": "SK하이닉스", "category": "비교"},
    {"question": "2026년 1분기 삼성전자 메모리 부문 영업이익은?", "keywords": ["2026", "메모리", "영업이익"], "expected_answer": "33.1조원", "category": "재무"},
    {"question": "2025년 4분기 DRAM ASP 상승률은 삼성과 SK 중 어느 쪽이 더 높은가?", "keywords": ["DRAM", "ASP", "삼성"], "expected_answer": "삼성전자", "category": "비교"},
]

print(f"테스트 질문 수: {len(test_questions)}")

테스트 질문 수: 12


In [5]:
def check_accuracy(llm_answer: str, expected_answer: str) -> str:
    """
    정확도: 전체 일치 또는 부분 일치 판정.
    반환: 'exact' | 'partial' | 'fail'
    """
    ans = llm_answer.strip()
    exp = expected_answer.strip()
    if not ans:
        return 'fail'
    if exp in ans:
        return 'exact'
    # 부분 일치: 숫자만 추출해서 비교 (93.8, 20.1 등)
    import re
    nums_exp = re.findall(r'[\d.,]+', exp)
    nums_ans = re.findall(r'[\d.,]+', ans)
    for n in nums_exp:
        if any(n in a for a in nums_ans):
            return 'partial'
    # 단어 일치 (SK하이닉스, 삼성전자 등)
    if exp in ans or any(w in ans for w in exp.split()):
        return 'partial'
    return 'fail'

In [6]:
def check_grounded(retrieved_docs: list, expected_answer: str) -> bool:
    """
    근거성: 인용된 문서에 정답이 포함되는지.
    retrieved_docs: Document 객체 리스트 (page_content 속성)
    """
    context = " ".join(d.page_content for d in retrieved_docs)
    return expected_answer.strip() in context

In [7]:
def evaluate_rag(question: str, expected_answer: str, retrieved_docs: list, llm_answer: str, elapsed_sec: float) -> dict:
    """
    RAG 단일 질문 평가.
    반환: {'accuracy': 'exact'|'partial'|'fail', 'grounded': True|False, 'elapsed': float}
    """
    acc = check_accuracy(llm_answer, expected_answer)
    grounded = check_grounded(retrieved_docs, expected_answer)
    return {"accuracy": acc, "grounded": grounded, "elapsed": elapsed_sec}

In [8]:
# 평가 함수 테스트 (샘플)
class _Doc:
    def __init__(self, content):
        self.page_content = content
sample_docs = [_Doc('삼성전자 2025년 4분기 매출액은 93.8조원이다.')]
ev = evaluate_rag(
    question="삼성전자 2025년 4분기 매출액은?",
    expected_answer="93.8조원",
    retrieved_docs=sample_docs,
    llm_answer="93.8조원입니다.",
    elapsed_sec=1.2
)
print(ev)

{'accuracy': 'exact', 'grounded': True, 'elapsed': 1.2}


In [9]:
# Part 2~5에서 사용: 질문셋 저장
import pickle
with open("test_questions_v2.pkl", "wb") as f:
    pickle.dump(test_questions, f)
print("test_questions 저장 완료.")

test_questions 저장 완료.
